# 📚 EduDocs AI

Agente conversacional basado en RAG utilizando:

- Google Gemini
- LangChain
- ChromaDB
- Google Colab

Proyecto desarrollado para el Challenge Alura ONE.

---

# 🎯 Objetivo

Desarrollar un agente de inteligencia artificial capaz de responder preguntas
sobre documentos de una plataforma educativa utilizando la arquitectura RAG
(Retrieval-Augmented Generation).

---

# 🏗️ Arquitectura

```text
Documentos
    │
    ▼
PyPDFLoader
    │
    ▼
Document
    │
    ▼
RecursiveCharacterTextSplitter
    │
    ▼
Chunks
    │
    ▼
Google Embeddings
    │
    ▼
ChromaDB
    │
    ▼
Retriever
    │
    ▼
Gemini
    │
    ▼
Respuesta
```

# ==========================================
# Colecta y organización
# ==========================================

En esta etapa se recopilaron y organizaron los documentos que formarán la base de conocimiento del agente.

Se creó una carpeta llamada **documents**, donde se almacenan archivos PDF, Word, Excel, PowerPoint, Markdown, CSV, JSON y HTML pertenecientes a una plataforma educativa.

# ==========================================
# Procesamiento y extracción
# ==========================================

En esta etapa los documentos son cargados, convertidos en objetos Document y posteriormente divididos en pequeños fragmentos (chunks) para facilitar la búsqueda semántica.

In [1]:
# =====================================
# Instalación de dependencias
# =====================================
# Instala todas las librerías necesarias para:
# - Construir el pipeline RAG con LangChain.
# - Procesar documentos de diferentes formatos.
# - Generar embeddings con Gemini.
# - Almacenar vectores en ChromaDB.
# - Leer archivos PDF, Word, Excel, HTML y PowerPoint.

!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-google-genai \
chromadb \
pypdf \
unstructured \
python-docx \
docx2txt \
openpyxl \
beautifulsoup4 \
python-pptx

In [2]:
# =====================================
# Importación de librerías
# =====================================

# Librerías estándar de Python
import json
from pathlib import Path

# Librerías para procesar documentos
from openpyxl import load_workbook      # Excel (.xlsx)
from bs4 import BeautifulSoup           # HTML
from pptx import Presentation           # PowerPoint (.pptx)

# Permite acceder de forma segura a la API Key almacenada en Google Colab
from google.colab import userdata

# Clase base utilizada por LangChain para representar documentos
from langchain_core.documents import Document

# Loaders de LangChain para cargar diferentes formatos de archivos
from langchain_community.document_loaders import (
    PyPDFLoader,        # PDF
    Docx2txtLoader,     # Word (.docx)
    TextLoader,         # Texto plano y Markdown
    CSVLoader,          # Archivos CSV
)

# Divide documentos grandes en fragmentos (chunks)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Modelo utilizado para convertir texto en embeddings mediante Google Gemini
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Base de datos vectorial utilizada para almacenar y buscar embeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_20967/1044755086.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


# =====================================
# Función para cargar documentos
# =====================================

In [3]:
# =====================================
# Función para cargar documentos
# =====================================

def cargar_documentos(carpeta):

    # Lista donde se almacenarán todos los documentos cargados
    documents = []

    # Convierte la ruta en un objeto Path para recorrer la carpeta
    carpeta = Path(carpeta)

    # Recorre todos los archivos de la carpeta
    for archivo in carpeta.iterdir():

        try:

            # Obtiene la extensión del archivo en minúsculas
            extension = archivo.suffix.lower()

            # ---------- PDF ----------
            if extension == ".pdf":

                loader = PyPDFLoader(str(archivo))

                documents.extend(loader.load())

                print(f"✔ PDF cargado: {archivo.name}")

            # ---------- WORD ----------
            elif extension == ".docx":

                loader = Docx2txtLoader(str(archivo))

                documents.extend(loader.load())

                print(f"✔ Word cargado: {archivo.name}")

            # ---------- MARKDOWN ----------
            elif extension == ".md":

                loader = TextLoader(
                    str(archivo),
                    encoding="utf-8"
                )

                documents.extend(loader.load())

                print(f"✔ Markdown cargado: {archivo.name}")

            # ---------- CSV ----------
            elif extension == ".csv":

                loader = CSVLoader(str(archivo))

                documents.extend(loader.load())

                print(f"✔ CSV cargado: {archivo.name}")

            # ---------- JSON ----------
            elif extension == ".json":

                # Lee el contenido del archivo JSON
                with open(archivo, "r", encoding="utf-8") as f:
                    contenido = json.load(f)

                # Convierte el JSON en texto legible
                texto = json.dumps(
                    contenido,
                    indent=2,
                    ensure_ascii=False
                )

                # Crea manualmente un objeto Document
                documents.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": str(archivo),
                            "tipo": "json"
                        }
                    )
                )

                print(f"✔ JSON cargado: {archivo.name}")

            # ---------- HTML ----------
            elif extension == ".html":

                # Lee el contenido HTML
                with open(archivo, "r", encoding="utf-8") as f:
                    html = f.read()

                # Elimina las etiquetas HTML y conserva únicamente el texto
                soup = BeautifulSoup(html, "html.parser")

                texto = soup.get_text(
                    separator="\n",
                    strip=True
                )

                # Crea un Document con el texto extraído
                documents.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": str(archivo),
                            "tipo": "html"
                        }
                    )
                )

                print(f"✔ HTML cargado: {archivo.name}")

            # ---------- POWERPOINT ----------
            elif extension == ".pptx":

                # Abre la presentación
                presentacion = Presentation(str(archivo))

                # Recorre todas las diapositivas
                for numero_slide, slide in enumerate(presentacion.slides, start=1):

                    texto = ""

                    # Extrae el texto de cada elemento de la diapositiva
                    for shape in slide.shapes:

                        if hasattr(shape, "text"):
                            texto += shape.text + "\n"

                    texto = texto.strip()

                    # Crea un Document únicamente si la diapositiva contiene texto
                    if texto:

                        documents.append(
                            Document(
                                page_content=texto,
                                metadata={
                                    "source": str(archivo),
                                    "tipo": "powerpoint",
                                    "slide": numero_slide
                                }
                            )
                        )

                print(f"✔ PowerPoint cargado: {archivo.name}")

            # ---------- EXCEL ----------
            elif extension == ".xlsx":

                # Abre el archivo Excel
                workbook = load_workbook(archivo)

                texto = ""

                # Recorre todas las hojas del libro
                for hoja in workbook.worksheets:

                    texto += f"\nHoja: {hoja.title}\n"

                    # Convierte cada fila en texto
                    for fila in hoja.iter_rows(values_only=True):

                        fila_texto = " | ".join(
                            str(celda)
                            for celda in fila
                            if celda is not None
                        )

                        texto += fila_texto + "\n"

                # Crea un Document con todo el contenido del Excel
                documents.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": str(archivo),
                            "tipo": "excel"
                        }
                    )
                )

                print(f"✔ Excel cargado: {archivo.name}")

            # ---------- OTROS ----------
            else:

                print(f"⏭ Archivo no soportado: {archivo.name}")

        # Captura cualquier error sin detener la carga del resto de documentos
        except Exception as e:

            print(f"❌ Error cargando {archivo.name}: {e}")

    # Muestra la cantidad total de documentos cargados
    print(f"\nTotal de documentos cargados: {len(documents)}")

    return documents

In [4]:
# Carga todos los documentos de la carpeta y los convierte en objetos Document
documents = cargar_documentos("documents")

✔ HTML cargado: Guia_de_Uso_de_la_Plataforma.html
✔ PDF cargado: Reglamento_del_Estudiante_6_paginas.pdf
✔ Word cargado: Politica_de_Reembolso_de_Matriculas.docx
✔ Excel cargado: Programa_de_Becas_y_Afiliados.xlsx
✔ Markdown cargado: Preguntas_Frecuentes_Cursos_y_Certificados.md

Total de documentos cargados: 11


In [5]:
# =====================================
# Verificar la cantidad de documentos cargados
# =====================================

len(documents)

11

In [6]:
# =====================================
# Visualizar el primer documento cargado
# =====================================

documents[0]

Document(metadata={'source': 'documents/Guia_de_Uso_de_la_Plataforma.html', 'tipo': 'html'}, page_content='Guía de Uso de la Plataforma\nGuía de Uso de la Plataforma Educativa\nEsta guía explica paso a paso cómo utilizar la plataforma educativa desde el registro hasta la obtención de certificados. Está dirigida a estudiantes nuevos y usuarios que desean aprovechar todas las funciones disponibles.\n1. Registro de una cuenta\nComplete el formulario de registro, verifique su correo electrónico y establezca una contraseña segura. Utilice información real para facilitar la emisión de certificados.\n2. Inicio de sesión\nAcceda con su correo y contraseña. Si olvidó sus credenciales, utilice la opción de recuperación de contraseña.\n3. Configuración del perfil\nActualice su nombre, fotografía, idioma, zona horaria y datos de contacto. Revise periódicamente que la información sea correcta.\n4. Explorar el catálogo\nUtilice los filtros por categoría, duración, nivel, precio e instructor para enc

In [7]:
# =====================================
# Configuración de la API Key
# =====================================

# Obtiene la API Key de Google Gemini almacenada de forma segura en Google Colab
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

In [8]:
# =====================================
# Dividir los documentos en chunks
# =====================================

# Configura el Text Splitter que se encargará de dividir
# los documentos en fragmentos más pequeños
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Tamaño máximo de cada chunk
    chunk_overlap=100     # Superposición entre chunks consecutivos
)

# Divide todos los documentos cargados en chunks
chunks = text_splitter.split_documents(documents)

# ==========================================
# Indexación
# ==========================================

Cada chunk es convertido en un embedding mediante Google Gemini y almacenado dentro de una base de datos vectorial ChromaDB.

In [9]:
# =====================================
# Crear el modelo de embeddings
# =====================================

# Inicializa el modelo de embeddings de Google Gemini, el cual
# convierte texto en vectores numéricos para realizar búsquedas semánticas
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=GOOGLE_API_KEY
)

In [10]:
# =====================================
# Crear o cargar la base de datos vectorial
# =====================================
# Verifica si ya existe una base vectorial almacenada localmente.
#
# - Si no existe, genera los embeddings de todos los chunks y crea
#   una nueva base de datos vectorial utilizando ChromaDB.
#
# - Si la base ya existe, la reutiliza para evitar volver a generar
#   los embeddings, reduciendo el tiempo de procesamiento y el consumo
#   de la cuota de la API de Gemini.
#
# - Si la base existe pero está vacía, elimina la carpeta y la crea
#   nuevamente para asegurar que todos los documentos queden indexados.

import os

if not os.path.exists("chroma_db"):

    print("Creando base vectorial...")

    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="chroma_db"
    )

else:

    vector_db = Chroma(
        persist_directory="chroma_db",
        embedding_function=embeddings
    )

    if vector_db._collection.count() == 0:

        print("La base existe pero está vacía. Recreando...")

        import shutil
        shutil.rmtree("chroma_db")

        vector_db = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory="chroma_db"
        )


/tmp/ipykernel_20967/669231891.py:30: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


In [11]:
# =====================================
# Verificar la indexación de los documentos
# =====================================

print(f"Chunks indexados: {vector_db._collection.count()}")

Chunks indexados: 79


# ==========================================
# Capa de Recuperación (RAG)
# ==========================================

En esta etapa el usuario realiza una pregunta.

LangChain convierte automáticamente esa pregunta en un embedding utilizando el mismo modelo empleado durante la indexación.

Posteriormente, ChromaDB compara ese embedding con todos los embeddings almacenados y recupera los fragmentos (chunks) más similares desde el punto de vista semántico.

Finalmente, estos fragmentos serán utilizados como contexto para que Gemini genere una respuesta en la siguiente etapa.

In [12]:
# =====================================
# Crear el Retriever
# =====================================

# Crea el componente encargado de recuperar los fragmentos
# más relevantes desde la base de datos vectorial utilizando
# la similitud entre la pregunta y los embeddings almacenados
retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 3   # Número de chunks que se recuperarán en cada búsqueda
    }
)

In [13]:
# =====================================
# Definir la pregunta del usuario
# =====================================

# Simula la consulta realizada por un usuario al agente RAG
pregunta = "¿Cómo obtengo mi certificado?"

In [14]:
# =====================================
# Recuperar información relevante
# =====================================

# Envía la pregunta al Retriever para recuperar los documentos
# más relevantes utilizando búsqueda semántica
resultados = retriever.invoke(pregunta)

In [15]:
# =====================================
# Mostrar los resultados de la búsqueda
# =====================================

# Recorre los documentos recuperados y muestra su contenido
# junto con el archivo del que fueron obtenidos
for i, doc in enumerate(resultados, start=1):

    # Imprime un encabezado para cada resultado
    print("=" * 60)
    print(f"Resultado {i}")
    print("=" * 60)

    # Muestra el contenido del fragmento recuperado
    print(doc.page_content)

    # Muestra el documento de origen del fragmento
    print("\nFuente:", doc.metadata["source"])

    print()

Resultado 1
Debes completar el 100 % del contenido, aprobar las evaluaciones y no
tener pagos pendientes.

## 10. ¿Cuándo se genera el certificado?

Normalmente de forma automática al cumplir todos los requisitos.

## 11. ¿El certificado tiene validez?

Cada certificado incluye un identificador único para verificar su
autenticidad.

## 12. ¿Puedo descargar mi certificado más de una vez?

Sí, estará disponible desde tu perfil mientras tu cuenta permanezca
activa.

## 13. ¿Qué hago si mi nombre aparece incorrecto?

Debes actualizar tu perfil y contactar al soporte para solicitar una
nueva emisión.

## 14. ¿Cómo recupero mi contraseña?

Utiliza la opción **Olvidé mi contraseña** en la pantalla de inicio de
sesión.

## 15. ¿Puedo cambiar mi correo electrónico?

Sí, desde la configuración de tu perfil.

Fuente: documents/Preguntas_Frecuentes_Cursos_y_Certificados.md

Resultado 2
# Preguntas Frecuentes sobre Cursos y Certificados

## Introducción

Este documento reúne las preguntas más comun

En esta etapa se implementó la recuperación de información (Retrieval).

Se creó un Retriever a partir de la base vectorial ChromaDB, el cual recibe una pregunta del usuario y recupera los fragmentos más relevantes utilizando búsqueda semántica.

Los fragmentos recuperados serán utilizados como contexto por Gemini en la siguiente etapa.

## Nota

En este proyecto no se implementó filtrado por metadatos ni un modelo de reranking, ya que se trata de un MVP con un único documento de una plataforma educativa.

La recuperación se realiza mediante búsqueda semántica utilizando ChromaDB y un Retriever de LangChain, lo cual es suficiente para cumplir

# ==========================================
# Generación de respuestas
# ==========================================

En esta etapa se conecta el Retriever con el modelo Gemini.

Cuando el usuario realiza una pregunta, el Retriever recupera los fragmentos más relevantes de la base vectorial.

Posteriormente, dichos fragmentos son enviados junto con la pregunta a Gemini para generar una respuesta basada únicamente en el contexto recuperado.

Finalmente, la respuesta se devuelve al usuario indicando la fuente de la información utilizada.

In [16]:
# =====================================
# Importar el modelo conversacional de Gemini
# =====================================

# Importa la clase utilizada para integrar un modelo
# conversacional de Google Gemini dentro de LangChain
from langchain_google_genai import ChatGoogleGenerativeAI

In [17]:
# =====================================
# Configurar el modelo de lenguaje (LLM)
# =====================================

# Configura el modelo de lenguaje de Google Gemini que generará
# las respuestas utilizando el contexto recuperado por el Retriever
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",          # Modelo de lenguaje
    google_api_key=GOOGLE_API_KEY,     # Credenciales de acceso a la API
    temperature=0                      # Reduce la aleatoriedad para obtener respuestas más consistentes
)

# ==========================================
# Construcción del Prompt
# ==========================================

El prompt define las instrucciones que seguirá Gemini para responder.

Se le indica que únicamente utilice el contexto recuperado por el Retriever, evitando generar información que no exista en los documentos.

Además, siempre deberá indicar la fuente utilizada y, si no encuentra la respuesta, informarlo explícitamente.

In [18]:
# =====================================
# Importar la plantilla de Prompt
# =====================================

# Importa la clase que permite construir el Prompt que recibirá
# el modelo de lenguaje junto con el contexto recuperado
from langchain_core.prompts import ChatPromptTemplate

In [19]:
# =====================================
# Crear el Prompt del agente
# =====================================

# Define las instrucciones que recibirá el modelo de lenguaje.
# El Prompt establece el comportamiento del agente indicando
# cómo debe responder y qué información puede utilizar.
prompt = ChatPromptTemplate.from_template("""
Eres un asistente virtual para una plataforma educativa.

Responde únicamente utilizando la información contenida en el contexto.

Si la respuesta no se encuentra en el contexto responde exactamente:

"No encontré esa información dentro de los documentos disponibles."

Nunca inventes información.

Si la información proviene de uno o varios documentos del contexto,
menciónalos al final de la respuesta cuando sea posible.

Contexto:
{context}

Pregunta:
{question}
""")

In [20]:
# =====================================
# Importar componentes para la cadena RAG
# =====================================

# Importa las clases que permiten construir el flujo de trabajo
# del agente y procesar la respuesta generada por el modelo
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [21]:
# =====================================
# Construir la cadena RAG
# =====================================

# Crea el flujo de trabajo del agente RAG. La pregunta del usuario
# se utiliza para recuperar el contexto relevante desde la base de
# datos vectorial, construir el Prompt, enviarlo al modelo Gemini
# y devolver la respuesta como texto.
rag_chain = (
    {
        # Recupera los fragmentos más relevantes desde ChromaDB
        "context": retriever,

        # Pasa la pregunta del usuario al Prompt
        "question": RunnablePassthrough()
    }

    # Construye el Prompt utilizando el contexto recuperado
    | prompt

    # Envía el Prompt al modelo de lenguaje
    | llm

    # Convierte la respuesta del modelo en texto plano
    | StrOutputParser()
)

In [23]:
# =====================================
# Consultar el agente
# =====================================

# Envía una pregunta al agente RAG y muestra la respuesta generada
respuesta = rag_chain.invoke(
    "¿Cómo obtengo mi certificado?"
)

print(respuesta)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 41.332658417s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

# =====================================
# Interfaz Web con Gradio
# =====================================

In [25]:
# =====================================
# Instalar Gradio
# =====================================
# Instala Gradio para crear una interfaz web sencilla
# que permita interactuar con el agente RAG.

!pip install -q gradio

In [26]:
# =====================================
# Función para consultar el agente
# =====================================

historial = []

def consultar_agente(pregunta):

    if not pregunta.strip():
        return "", "\n".join(historial)

    respuesta = rag_chain.invoke(pregunta)

    historial.append(
        f"👤 Usuario:\n{pregunta}\n\n🤖 EduDocsAI:\n{respuesta}"
    )

    historial_texto = "\n\n" + ("=" * 70 + "\n\n").join(historial)

    return respuesta, historial_texto

In [27]:
# =====================================
# Crear la interfaz con Gradio
# =====================================

import gradio as gr

# -------------------------------------
# Funciones para los botones de feedback
# -------------------------------------

def feedback_positivo():
    return "¡Gracias por tu retroalimentación!"

def feedback_negativo():
    return "Gracias. La respuesta ayudará a mejorar el agente."

# -------------------------------------
# Interfaz principal
# -------------------------------------

with gr.Blocks(title="📚 EduDocsAI - Asistente Inteligente para Documentos Educativos") as demo:

    gr.Markdown("""
# 📚 EduDocsAI

### Asistente Virtual basado en Inteligencia Artificial

Este agente responde preguntas utilizando únicamente la información contenida
en los documentos cargados en el sistema.

Escribe una pregunta y el agente buscará la información más relevante utilizando
la base vectorial creada con LangChain, ChromaDB y Google Gemini.
""")

    # ---------------------------------
    # Pregunta del usuario
    # ---------------------------------

    pregunta = gr.Textbox(
        label="Escribe tu pregunta",
        placeholder="Ejemplo: ¿Cómo obtengo mi certificado?"
    )

    # ---------------------------------
    # Respuesta del agente
    # ---------------------------------

    respuesta = gr.Textbox(
        label="Respuesta",
        lines=10,
        interactive=False
    )

    # ---------------------------------
    # Historial de conversación
    # ---------------------------------

    historial_chat = gr.Textbox(
        label="Historial de conversación",
        lines=12,
        interactive=False
    )

    # ---------------------------------
    # Botón para consultar al agente
    # ---------------------------------

    boton = gr.Button(
        "🚀 Consultar",
        variant="primary"
    )

    boton.click(
        fn=consultar_agente,
        inputs=pregunta,
        outputs=[
            respuesta,
            historial_chat
        ]
    )

    # ---------------------------------
    # Retroalimentación del usuario
    # ---------------------------------

    gr.Markdown("## 👍 Retroalimentación")

    gr.Markdown(
        "¿La respuesta proporcionada por el agente fue útil?"
    )

    with gr.Row():

        boton_si = gr.Button("👍 Sí")

        boton_no = gr.Button("👎 No")

    feedback = gr.Textbox(
        label="Estado",
        interactive=False
    )

    boton_si.click(
        fn=feedback_positivo,
        outputs=feedback
    )

    boton_no.click(
        fn=feedback_negativo,
        outputs=feedback
    )

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e68790feb130c9fe5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
